# GRPO Rollout Generation & Dataset Creation

Generates self-correction rollouts on the MBPP train split with an SFT model, then builds a recovery-only GRPO dataset.

In [ ]:
import re
import torch; v = re.match(r"[0-9]{1,}\.[0-9]{1,}", str(torch.__version__)).group(0)
xformers = "xformers==" + ("0.0.33.post1" if v=="2.9" else "0.0.32.post2" if v=="2.8" else "0.0.29.post3")
!pip install --no-deps bitsandbytes accelerate {xformers} peft trl triton cut_cross_entropy unsloth_zoo
!pip install sentencepiece protobuf "datasets==4.3.0" "huggingface_hub>=0.34.0" hf_transfer
!pip install --no-deps unsloth
!pip install transformers==4.56.2
!pip install --no-deps trl==0.22.2

In [ ]:
import os
import re
import io
import sys
import ast
import json
import ctypes
import random
import threading
import contextlib
import traceback
import resource
import concurrent.futures
from collections import Counter

import torch
from datasets import load_dataset, Dataset
from unsloth import FastLanguageModel

os.environ["PYTORCH_CUDA_ALLOC_CONF"] = "expandable_segments:True"

## 1. Model loading

In [ ]:
import huggingface_hub

huggingface_hub.login(token=os.environ["HF_TOKEN"])

In [ ]:
MODEL_NAME  = "moazeldegwy/Qwen3-0.6B-LABD-2.1-SFT"
MAX_SEQ_LEN = 6144

TEMPERATURE        = 0.6
TOP_P              = 0.95
TOP_K              = 20
MIN_P              = 0.0
REPETITION_PENALTY = 1.05

MAX_NEW_TOKENS = 6144
MAX_INPUT_LEN  = 6144

model, tokenizer = FastLanguageModel.from_pretrained(
    MODEL_NAME,
    max_seq_length = MAX_SEQ_LEN,
    load_in_4bit   = False,
)

tokenizer.padding_side    = "left"
tokenizer.truncation_side = "left"
if tokenizer.pad_token is None:
    tokenizer.pad_token = tokenizer.eos_token

FastLanguageModel.for_inference(model)

## 2. Sandbox (thread-safe)

In [ ]:
class TimeoutException(Exception):
    pass

def timeout_handler(signum, frame):
    raise TimeoutException

def _async_raise_in_thread(tid, exc_type):
    res = ctypes.pythonapi.PyThreadState_SetAsyncExc(
        ctypes.c_ulong(tid), ctypes.py_object(exc_type)
    )
    if res == 0:
        return False
    if res > 1:
        ctypes.pythonapi.PyThreadState_SetAsyncExc(ctypes.c_ulong(tid), None)
        return False
    return True

def analyze_assertion_failure(line_code, context):
    try:
        tree = ast.parse(line_code.strip())
        if not isinstance(tree.body[0], ast.Assert):
            return None
        assert_node = tree.body[0]
        node = assert_node.test
        if isinstance(node, ast.Compare):
            left_node  = node.left
            right_node = node.comparators[0]
            left_val   = eval(compile(ast.Expression(body=left_node),  filename="<string>", mode="eval"), context)
            right_val  = eval(compile(ast.Expression(body=right_node), filename="<string>", mode="eval"), context)
            return {
                "actual":         left_val,
                "expected":       right_val,
                "actual_repr":    ast.unparse(left_node),
                "expected_repr":  ast.unparse(right_node),
            }
    except Exception:
        return None
    return None

def execute_code_with_feedback(code_str, cases=None, timeout_seconds=10, memory_limit_mb=4096):
    if cases is None:
        cases = []
    stdout_capture = io.StringIO()
    stderr_capture = io.StringIO()
    full_execution_script = f"{code_str}\n\n" + "\n".join(cases)

    target_tid = threading.get_ident()
    timer = threading.Timer(
        timeout_seconds,
        lambda: _async_raise_in_thread(target_tid, TimeoutException),
    )
    timer.daemon = True
    timer.start()

    apply_mem_limit = threading.current_thread() is threading.main_thread()
    soft = hard = None
    if apply_mem_limit:
        soft, hard = resource.getrlimit(resource.RLIMIT_AS)
        try:
            resource.setrlimit(resource.RLIMIT_AS, (memory_limit_mb * 1024 * 1024, hard))
        except ValueError:
            apply_mem_limit = False

    local_scope = {}
    try:
        with contextlib.redirect_stdout(stdout_capture), contextlib.redirect_stderr(stderr_capture):
            exec(full_execution_script, local_scope)
        timer.cancel()
        return True, f"Execution Successful. Stdout:\n{stdout_capture.getvalue()}", "None"

    except TimeoutException:
        timer.cancel()
        return False, "Execution Failed: Time Limit Exceeded (Possible Infinite Loop)", "Runtime"
    except MemoryError:
        timer.cancel()
        return False, "Execution Failed: Memory Limit Exceeded", "Runtime"
    except Exception:
        timer.cancel()
        exc_type, exc_value, exc_traceback = sys.exc_info()
        exc_name = exc_type.__name__
        tb_frames = traceback.extract_tb(exc_traceback)
        error_line_num = "Unknown"
        offending_line_code = "Could not extract line."
        if hasattr(exc_value, "lineno"):
            error_line_num = exc_value.lineno
        elif tb_frames:
            for frame in reversed(tb_frames):
                if frame.filename == "<string>":
                    error_line_num = frame.lineno
                    break
        if isinstance(error_line_num, int):
            script_lines = full_execution_script.splitlines()
            if 0 <= error_line_num - 1 < len(script_lines):
                offending_line_code = script_lines[error_line_num - 1].strip()
        details = str(exc_value)
        rich_feedback = ""
        if exc_name == "AssertionError":
            analysis = analyze_assertion_failure(offending_line_code, local_scope)
            if analysis:
                act_str = str(analysis["actual"])
                exp_str = str(analysis["expected"])
                if len(act_str) > 500: act_str = act_str[:500] + "... (truncated)"
                if len(exp_str) > 500: exp_str = exp_str[:500] + "... (truncated)"
                rich_feedback = (
                    f"\nAnalysis:\n   Input/Call: {analysis['actual_repr']}\n"
                    f"   Expected: {exp_str}\n   Actual:    {act_str}"
                )
        error_msg = f"Error Type: {exc_name}\nLine {error_line_num}: {offending_line_code}\nDetails: {details}{rich_feedback}"
        return False, error_msg, "Logical" if exc_name == "AssertionError" else "Runtime"
    finally:
        timer.cancel()
        if apply_mem_limit and soft is not None:
            try:
                resource.setrlimit(resource.RLIMIT_AS, (soft, hard))
            except ValueError:
                pass

## 3. Self-correction agent (batched rollouts)

In [ ]:
SYSTEM_PROMPT = """You are a Self-Correcting Python Agent. Solve the problem; the sandbox tool will verify your code.
### OUTPUT FORMAT
After your reasoning, output ONE <execute> block containing the function plus the user's assert statements appended verbatim. Nothing else after it.
Example:
<execute>
def add(a, b):
    return a + b
assert add(2, 3) == 5
</execute>
### RULES
- Exactly one <execute> block per turn. Function and asserts together, never split into separate blocks.
- On a debugging turn: quote Expected vs Actual from the sandbox tool feedback in your reasoning and identify the root cause before writing the corrected <execute> block.
"""


class BatchedSelfCorrectionAgent:
    def __init__(self, model, tokenizer):
        self.model = model
        self.tokenizer = tokenizer

    @staticmethod
    def extract_code(response_text):
        m = re.search(r"<execute>(.*?)</execute>", response_text, re.DOTALL)
        return m.group(1).strip() if m else None

    def run_batched(self, dataset, batch_size=2, max_retries=3, output_file="mbpp_train_rollouts.jsonl"):
        processed_ids = set()
        if os.path.exists(output_file):
            with open(output_file, "r", encoding="utf-8") as f:
                for line in f:
                    line = line.strip()
                    if not line:
                        continue
                    try:
                        rec = json.loads(line)
                        if "task_id" in rec:
                            processed_ids.add(rec["task_id"])
                    except json.JSONDecodeError:
                        pass

        active_tasks = [t for t in dataset if t["task_id"] not in processed_ids]
        total = len(active_tasks)

        for i in range(0, total, batch_size):
            batch = active_tasks[i:i+batch_size]
            self._process_batch(batch, max_retries, output_file)

    def _process_batch(self, batch, max_retries, output_file):
        states = []
        for task in batch:
            user_msg = f"Problem: {task['prompt']}\n\nTest Cases:\n" + "\n".join(task["tests"])
            states.append({
                "task_id":  task["task_id"],
                "tests":    task["tests"],
                "messages": [
                    {"role": "system", "content": SYSTEM_PROMPT},
                    {"role": "user",   "content": user_msg},
                ],
                "attempts": 0,
                "solved":   False,
                "done":     False,
            })

        while True:
            active = [s for s in states if not s["done"]]
            if not active:
                break

            texts = [
                self.tokenizer.apply_chat_template(
                    s["messages"],
                    tokenize=False,
                    add_generation_prompt=True,
                    enable_thinking=True,
                )
                for s in active
            ]
            inputs = self.tokenizer(
                texts,
                return_tensors="pt",
                padding=True,
                truncation=True,
                max_length=MAX_INPUT_LEN,
            ).to("cuda")

            gen_kwargs = dict(
                max_new_tokens     = MAX_NEW_TOKENS,
                do_sample          = True,
                temperature        = TEMPERATURE,
                top_p              = TOP_P,
                top_k              = TOP_K,
                min_p              = MIN_P,
                repetition_penalty = REPETITION_PENALTY,
                pad_token_id       = self.tokenizer.pad_token_id,
            )

            with torch.inference_mode():
                generated_ids = self.model.generate(**inputs, **gen_kwargs)
            new_tokens = generated_ids[:, inputs.input_ids.shape[1]:]
            responses = self.tokenizer.batch_decode(new_tokens, skip_special_tokens=True)

            max_workers = min(len(active), os.cpu_count() or 4)
            with concurrent.futures.ThreadPoolExecutor(max_workers=max_workers) as ex:
                futures = []
                for idx, resp in enumerate(responses):
                    code_str = self.extract_code(resp)
                    if code_str:
                        futures.append(ex.submit(execute_code_with_feedback, code_str, active[idx]["tests"]))
                    else:
                        futures.append(None)

                results = []
                for f in futures:
                    if f is None:
                        results.append((False, "Execution Failed: No <execute> block found in response.", "Parsing"))
                    else:
                        try:
                            results.append(f.result(timeout=15))
                        except Exception as e:
                            results.append((False, f"Execution Failed: {e}", "Runtime"))

            for idx, (success, output_msg, error_type) in enumerate(results):
                state = active[idx]
                state["attempts"] += 1
                state["messages"].append({"role": "assistant", "content": responses[idx]})

                if success:
                    state["solved"] = True
                    state["done"]   = True
                    state["messages"].append({
                        "role": "user",
                        "content": f"<tool_response>\nExecution Success\n{output_msg}\n</tool_response>",
                    })
                else:
                    state["messages"].append({
                        "role": "user",
                        "content": f"<tool_response>\nExecution Failed\n{output_msg}\n</tool_response>",
                    })
                    if state["attempts"] >= max_retries:
                        state["done"] = True

                if state["done"]:
                    record = {
                        "task_id":       state["task_id"],
                        "solved":        state["solved"],
                        "attempts_used": state["attempts"],
                        "conversation":  state["messages"],
                    }
                    with open(output_file, "a", encoding="utf-8") as f:
                        f.write(json.dumps(record) + "\n")

            torch.cuda.empty_cache()

## 4. Run rollouts on MBPP train split

In [ ]:
from google.colab import drive
drive.mount('/content/drive')

In [ ]:
ds = load_dataset("google-research-datasets/mbpp")
train_data = ds["train"].to_list()

prepared = [
    {
        "task_id": row["task_id"],
        "prompt":  row["text"],
        "tests":   row["test_list"],
    }
    for row in train_data[:125]
]

ROLLOUT_FILE = "/content/drive/MyDrive/LABD2/mbpp_train_rollouts_grpo_0.6B.jsonl"

torch.cuda.empty_cache()
agent = BatchedSelfCorrectionAgent(model, tokenizer)
agent.run_batched(
    prepared,
    batch_size  = 10,
    max_retries = 3,
    output_file = ROLLOUT_FILE,
)
torch.cuda.empty_cache()

## 5. Build GRPO dataset (recovery-only, cap 2 per task)

In [ ]:
def prepare_grpo_from_rollouts(input_file, cap_per_task=2, include_first_attempt=True):
    """Build GRPO training prompts from rollout conversations.

    First-attempt prompts (attempt_index=0) use only [system, user]; recovery
    prompts (attempt_index=1..cap) include the prior failures and tool_response.
    """
    training_data = []

    with open(input_file, "r", encoding="utf-8") as f:
        for line in f:
            entry = json.loads(line)
            task_id = entry.get("task_id")
            conv    = entry.get("conversation", [])
            if len(conv) < 2:
                continue

            parts = conv[1]["content"].split("Test Cases:\n")
            test_cases = []
            if len(parts) > 1:
                test_cases = [ln for ln in parts[1].strip().split("\n") if ln.strip()]

            if include_first_attempt:
                first_attempt_prompt = tokenizer.apply_chat_template(
                    conv[:2],
                    tokenize              = False,
                    add_generation_prompt = True,
                    enable_thinking       = True,
                )
                training_data.append({
                    "prompt":              first_attempt_prompt,
                    "answer":              test_cases,
                    "task_id":             task_id,
                    "attempt_index":       0,
                    "prior_was_truncated": False,
                    "prior_error_type":    "None",
                })

            task_examples = []
            failure_counter = 0
            for i, msg in enumerate(conv):
                if msg["role"] != "user":
                    continue
                if "<tool_response>\nExecution Failed" not in msg["content"]:
                    continue

                failure_counter += 1
                history = conv[:i+1]

                formatted_prompt = tokenizer.apply_chat_template(
                    history,
                    tokenize              = False,
                    add_generation_prompt = True,
                    enable_thinking       = True,
                )

                prior_assistant = (
                    conv[i-1]["content"]
                    if i > 0 and conv[i-1]["role"] == "assistant"
                    else ""
                )
                prior_was_truncated = (
                    "<think>" in prior_assistant and "</think>" not in prior_assistant
                )
                fb = msg["content"]
                if "No <execute> block" in fb or "No valid code block" in fb:
                    prior_error_type = "Parsing"
                elif "AssertionError" in fb:
                    prior_error_type = "Logical"
                else:
                    prior_error_type = "Runtime"

                task_examples.append({
                    "prompt":              formatted_prompt,
                    "answer":              test_cases,
                    "task_id":             task_id,
                    "attempt_index":       failure_counter,
                    "prior_was_truncated": prior_was_truncated,
                    "prior_error_type":    prior_error_type,
                })

            training_data.extend(task_examples[:cap_per_task])

    random.shuffle(training_data)
    return training_data

ROLLOUT_FILE = "/mnt/train/mbpp_train_rollouts_grpo.jsonl"

raw_train_data = prepare_grpo_from_rollouts(
    ROLLOUT_FILE,
    cap_per_task          = 2,
    include_first_attempt = True,
)

train_dataset = Dataset.from_list(raw_train_data)
print(f"Rows: {len(train_dataset)}")
print("By attempt_index:   ", Counter(train_dataset["attempt_index"]))
print("By prior_error_type:", Counter(train_dataset["prior_error_type"]))
print("Truncated priors:   ", sum(train_dataset["prior_was_truncated"]))

## 6. Save dataset

In [ ]:
DRIVE_DIR = "/mnt/train"
os.makedirs(DRIVE_DIR, exist_ok=True)

dataset_path = os.path.join(DRIVE_DIR, "grpo_train_dataset-2.1")
train_dataset.save_to_disk(dataset_path)

In [ ]:
!zip -r grpo_train_dataset2.zip /mnt/train/grpo_train_dataset-2_1